# **Parallel Chain**

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI



load_dotenv()


if os.environ.get("OPENAI_API_KEY") :
    print("Bro API key variable is found")

else :
    raise ValueError("OPENAI_APKI_KEY not found")

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

Bro API key variable is found


In [49]:
from typing import Literal
from pydantic import BaseModel

class llm_schema(BaseModel) :
    movie_summary_flag : Literal["positive", "negative"]

llm_structured_ouput = llm_openai.with_structured_output(llm_schema)


# result = llm_structured_ouput.invoke("The movie was awesome")
# result


## **Conditional Chain**

In [44]:
#TASK-1 Prompt
prompt_template = ChatPromptTemplate(
    [
    ("system", "You are a movie review writer"),
    ("human","PLease categorize the movie review as positive or negative : {input}")
    ]
)

In [45]:
#Task -2 LLM
llm_structured_ouput = llm_openai.with_structured_output(llm_schema)

In [54]:
# Task - 3 Custom Runnable
from langchain_core.runnables import RunnableLambda

def pydantic_json(input:llm_schema)->str :
    return input.model_dump()['movie_summary_flag']

# result = llm_structured_ouput.invoke("The movie was awesome")
# res = pydantic_json(result)
# res

pydantic_json_lambda = RunnableLambda(pydantic_json)


### **Conditional Chain 1**

In [27]:
#Task - 1 Prompt

message_prompt = ChatPromptTemplate.from_messages(
    messages=[
    ("system", "You are message reply generator"),
    ("human","Create a short reply either thanking or apologising for the review : {text}")
    ]
)

#Task - 2 LLM
#Task - 3 Str Parser 
str_parser = StrOutputParser()
chain_positive = message_prompt | llm_openai | str_parser

### **Conditional Chain 2**

In [22]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

In [66]:
def insta_chain(text : str):

    text = text
    #Task - 1 Prompt

    insta_prompt = ChatPromptTemplate.from_messages(
        messages=[
        ("system", "You are message reply generator"),
        ("human","Create a short reply apologising for the review : {text}")
    ]
    )

    #Task - 2 LLM
    #Task - 3 String Parser 
    str_parser = StrOutputParser()
    
    chain_insta = insta_prompt | llm_openai | str_parser
    result = chain_insta.invoke(text)

    return (result)


negative_chain_runnable = RunnableLambda(insta_chain)




### **Final Orchestration**

In [67]:


condtional_chain = RunnableBranch(
    (lambda x:"positive" in x, chain_positive),
    negative_chain_runnable
    # default_chain
)

final_orchestrator = prompt_template|llm_structured_ouput|pydantic_json_lambda|condtional_chain

In [68]:
final_orchestrator.invoke({"input" : "It was a waste of time movie"})

'Hi — I’m very sorry to hear about your experience. We take feedback seriously and would like to make this right; please DM us or email [support@yourcompany.com] so we can resolve it promptly.'